#### Purpose: Ingest raw graduate student longitudinal data, flag and fix quality issues, construct metrics, and export a clean parquet file for reporting and modeling.

#### Git commit history for this file:
##### Commit 1: Skeleton and raw loading of data (done)
##### Commit 2: Standardize data - cleaning (done)
##### Commit 3: Construct financial support and career metrics (done)
##### Commit 4: add data-quality report export (done)

In [1]:
# importing packages
import pandas as pd
import numpy as np
from pathlib import Path
import json

### PATHS to relative project root

In [2]:
# This notebook lives in /src, so go up one level to reach the project root.
PROJECT_ROOT = Path().resolve().parent

RAW_PATH   = PROJECT_ROOT / "data" / "raw" / "grad_students_raw.csv"
CLEAN_PATH = PROJECT_ROOT / "data" / "processed" / "grad_students_clean.parquet"
QA_PATH    = PROJECT_ROOT / "data" / "processed" / "data_quality_report.json"

print("Project root resolved to:", PROJECT_ROOT)
print("Raw data path:", RAW_PATH)

Project root resolved to: /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis
Raw data path: /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/raw/grad_students_raw.csv


### Step 1: Loading data

In [3]:
## using Parquet for efficiency and reduction of column type errors
def load_raw(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)  # reads everything as string first, decide types later
    print(f"[LOAD] {len(df):,} x {len(df.columns)} columns loaded from {path}")
    return df

### Step 2: Standardizing data

In [4]:
BOOL_COLUMNS = ["pell_eligible", "TA_status", "GSR_status", "first_gen", "international"]

def _to_bool(series: pd.Series) -> pd.Series:
    """Convert free-text yes/no column to nullable boolean (pd.BooleanDtype)."""
    mapping = {"yes": True,
               "no": False,
               "1": True,
               "0": False,
               "true": True,
               "false": False}
    cleaned = series.str.strip().str.lower().map(mapping)
    return cleaned.astype(pd.BooleanDtype())

def standardize_booleans(df: pd.DataFrame) -> pd.DataFrame:
    """Apply bool normalization to all flag columns."""
    for col in BOOL_COLUMNS:
        df[col] = _to_bool(df[col])
    print(f"[STANDARDIZE] Boolean columns normalized")
    return df

def standardize_numerics(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cast numeric columns; coerce unparseable strings to NaN so we know
    exactly where the gaps are rather than silently dropping rows.
    """
    numeric_cols = {
        "grant_amt":          float,
        "gpa":                float,
        "units_enrolled":     int,
        "stipend_amt":        float,
        "time_to_degree":     float,
        "salary_offer":       float,
        "months_to_employment": float,
        "year_in_program":    int,
        "term_year":          int,
    }
    for col, dtype in numeric_cols.items():
        df[col] = pd.to_numeric(df[col], errors="coerce")
        if dtype == int:
            df[col] = df[col].astype("Int64")   # nullable integer
        else:
            df[col] = df[col].astype(float)
    print(f"[STANDARDIZE] Numeric columns cast")
    return df

def standardize_strings(df: pd.DataFrame) -> pd.DataFrame:
    """Title-case free-text columns; strip whitespace."""
    str_cols = ["program", "thesis_status", "employment_outcome",
                "employer_type", "job_title", "industry", "gender",
                "ethnicity", "academic_term"]
    for col in str_cols:
        df[col] = df[col].str.strip().str.title()
    df["last_name"]  = df["last_name"].str.strip().str.upper()
    df["first_name"] = df["first_name"].str.strip().str.title()
    print(f"[STANDARDIZE] String columns cleaned")
    return df

In [5]:
df = load_raw(RAW_PATH)
df = standardize_booleans(df)
df = standardize_numerics(df)
df = standardize_strings(df)

original_columns = set(df.columns)

[LOAD] 550 x 28 columns loaded from /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/raw/grad_students_raw.csv
[STANDARDIZE] Boolean columns normalized
[STANDARDIZE] Numeric columns cast
[STANDARDIZE] String columns cleaned


### Step 3: Creating metrics
#### new columns created
- term_label: human-readable "Fall 2019"
- any_support: TA or GSR in that term
- financial_aid_flag: pell_eligible OR grant_amt > 0
- total_financial_support: grant_amt + stipend_amt
- gpa_risk_flag: GPA below 3.0 (probation threshold)
- thesis_stage_num: ordinal thesis progress
- degree_program_type: PhD or MS
- cohort_year: first term_year a student appears
- year_since_cohort: term_year minus cohort_year
- pell_x_first_gen: interaction of both pell-eligible AND first generation status
- graduated_flag: True if the student has a non-null time_to_degree (i.e. has graduated)
- log_salary: log-transformed salary for modeling
- employed_flag: binary outcome for classification modeling (NA if not yet graduated)

#### Exploring each metric one at a time below before consolidating into a single `engineer_features()` function.

In [6]:
# Term Labels
df["term_label"] = df["academic_term"].str.strip() + " " + df["term_year"].astype(str)

In [7]:
# Support flags
df["any_support"] = (df["TA_status"].fillna(False) |
                        df["GSR_status"].fillna(False))

df["financial_aid_flag"] = (
     df["pell_eligible"].fillna(False) |
     (df["grant_amt"].fillna(0) > 0)
    )

df["total_financial_support"] = (
    df["grant_amt"].fillna(0) + df["stipend_amt"].fillna(0)
)

In [8]:
# Academic risk
df["gpa_risk_flag"] = df["gpa"] < 3.0

In [9]:
# Thesis Stage (ordinal)
stage_order = {
    "Proposal": 1, "Approved": 2,
    "Writing": 3, "Defended": 4,
    "N/A": 0
}
df["thesis_stage_num"] = (
    df["thesis_status"].str.title().map(stage_order).fillna(0).astype(int)
)

In [10]:
# Degree Type
df["degree_program_type"] = np.where(
    df["program"].str.upper().str.startswith("PHD"),
    "PHD",
    "Masters"
)

In [11]:
# Cohort Year
df["cohort_year"] = df.groupby("student_id")["term_year"].transform("min")
df["years_since_cohort"] = df["term_year"].astype(int) - df["cohort_year"]

In [12]:
# Interaction pell and first gen
df["pell_x_first_gen"] = (
     df["pell_eligible"].fillna(False) & df["first_gen"].fillna(False)
).astype(int)

In [13]:
# Career outcomes
# Define graduation status explicitly: a student is "graduated" if they have
# a non-null time_to_degree.
df["graduated_flag"] = df["time_to_degree"].notna()

# employed_flag is only meaningful for graduates:
#   1   = graduated AND employed
#   0   = graduated AND NOT employed (seeking, stopped out, etc.)
#   <NA> = not yet graduated (question doesn't apply yet)
df["employed_flag"] = pd.NA
df.loc[df["graduated_flag"] & (df["employment_outcome"].str.upper() == "EMPLOYED"), "employed_flag"] = 1
df.loc[df["graduated_flag"] & (df["employment_outcome"].str.upper() != "EMPLOYED"), "employed_flag"] = 0
df["employed_flag"] = df["employed_flag"].astype("Int64")

# Salary should only exist for graduates; force non-graduates to NaN
# regardless of what the raw data happened to contain.
df.loc[~df["graduated_flag"], "salary_offer"] = np.nan
df["log_salary"] = np.log(df["salary_offer"].replace(0, np.nan))

# Confirm what got added, using simple set arithmetic instead of inspecting
# another function's bytecode (the old version used load_raw.__code__.co_varnames,
# which only happened to work by coincidence and didn't actually count new
# features correctly).
new_features = sorted(set(df.columns) - original_columns)
print(f"[ENGINEER] {len(new_features)} new features created: {new_features}")

[ENGINEER] 13 new features created: ['any_support', 'cohort_year', 'degree_program_type', 'employed_flag', 'financial_aid_flag', 'gpa_risk_flag', 'graduated_flag', 'log_salary', 'pell_x_first_gen', 'term_label', 'thesis_stage_num', 'total_financial_support', 'years_since_cohort']


### Sanity checks before moving on

In [14]:
## testing
print(df.columns.tolist())

['student_id', 'last_name', 'first_name', 'academic_term', 'term_year', 'program', 'year_in_program', 'degree_type', 'pell_eligible', 'grant_amt', 'fellowship', 'TA_status', 'GSR_status', 'gpa', 'units_enrolled', 'stipend_amt', 'thesis_status', 'time_to_degree', 'employment_outcome', 'employer_type', 'salary_offer', 'job_title', 'industry', 'months_to_employment', 'gender', 'ethnicity', 'first_gen', 'international', 'term_label', 'any_support', 'financial_aid_flag', 'total_financial_support', 'gpa_risk_flag', 'thesis_stage_num', 'degree_program_type', 'cohort_year', 'years_since_cohort', 'pell_x_first_gen', 'graduated_flag', 'employed_flag', 'log_salary']


In [15]:
# Sub-dataset: one row per graduate, with career outcomes
graduates = df[df["graduated_flag"]].copy()

assert graduates["student_id"].is_unique, "Duplicate student_id found in graduates!"

graduates = graduates[[
    "student_id", "program", "degree_type", "time_to_degree",
    "graduated_flag", "employment_outcome", "employed_flag",
    "employer_type", "salary_offer",
    "job_title", "industry", "months_to_employment"
]].reset_index(drop=True)

print(f"[OK] {len(graduates)} graduates, one row each.")
graduates

[OK] 42 graduates, one row each.


,student_id,program,degree_type,time_to_degree,graduated_flag,employment_outcome,employed_flag,employer_type,salary_offer,job_title,industry,months_to_employment
0,1008,Ms Business Analytics,MS,2.0,True,Employed,1,Industry,112000.0,Engineer,Tech,1.3
1,1009,Ms Statistics,MS,1.2,True,Employed,1,Industry,114000.0,Data Scientist,Tech,4.6
2,1014,Ma Higher Education,MA,2.7,True,Employed,1,Academia,40000.0,Visiting Scholar,Higher Education,9.0
3,1016,Ma Higher Education,MA,2.9,True,Employed,1,Academia,45000.0,Postdoctoral Researcher,Higher Education,5.9
4,1018,Phd Public Health,PhD,5.8,True,Employed,1,Academia,73000.0,Visiting Scholar,Higher Education,6.1
5,1022,Mph,MPH,0.9,True,Employed,1,Government,71000.0,Economist,Government,12.0
6,1023,Mph,MPH,1.3,True,Employed,1,Government,76000.0,Research Analyst,Government,9.8
7,1024,Ma Counseling,MA,2.3,True,Employed,1,Nonprofit,42000.0,Development Officer,Nonprofit,6.9
8,1029,Phd Engineering,PhD,3.8,True,Employed,1,Consulting,105000.0,Management Consultant,Consulting,0.6
9,1030,Ms Statistics,MS,1.1,True,Employed,1,Industry,114000.0,Data Scientist,Tech,8.3


In [16]:
print(graduates["employed_flag"].value_counts(dropna=False))
print()
print(graduates["employment_outcome"].value_counts(dropna=False))

employed_flag
1    41
0     1
Name: count, dtype: Int64

employment_outcome
Employed              41
Seeking Employment     1
Name: count, dtype: int64


In [17]:
print(pd.crosstab(graduates["employment_outcome"], graduates["employed_flag"]))

employed_flag       0   1
employment_outcome       
Employed            0  41
Seeking Employment  1   0


## Data quality report output

In [18]:
def build_quality_report(df: pd.DataFrame) -> dict:
    """Produce a JSON-serializable dict describing the data quality state
    AFTER cleaning. This becomes the data_quality_report.json artifact.

    NOTE: this function computes its own `graduates` subset internally,
    rather than relying on a `graduates` variable defined elsewhere in the
    notebook. That keeps it self-contained so it works correctly whether
    called interactively or from main().
    """
    graduates = df[df["graduated_flag"]].copy()

    report = {
        "total_rows":    int(len(df)),
        "total_students": int(df["student_id"].nunique()),
        "programs":      df["program"].unique().tolist(),
        "term_range":    {
            "min": str(df["term_label"].min()),
            "max": str(df["term_label"].max()),
        },
        "null_counts": {
            col: int(df[col].isna().sum())
            for col in df.columns
        },
        "gpa_stats": {
            "mean":  round(float(df["gpa"].mean()), 3),
            "std":   round(float(df["gpa"].std()), 3),
            "min":   round(float(df["gpa"].min()), 3),
            "max":   round(float(df["gpa"].max()), 3),
        },
        "pct_pell_eligible":  round(float(df["pell_eligible"].mean()) * 100, 1),
        "pct_first_gen":      round(float(df["first_gen"].mean()) * 100, 1),
        "pct_any_support":    round(float(df["any_support"].mean()) * 100, 1),
        "employment_rate_pct": round(
            float(graduates["employed_flag"].mean()) * 100, 1
        ),
    }
    return report

In [19]:
# Quick check that build_quality_report works on its own before running the
# full pipeline via main()
report = build_quality_report(df)
report

{'total_rows': 550,
 'total_students': 95,
 'programs': ['Phd Chemistry',
  'Phd Sociology',
  'Ms Business Analytics',
  'Phd Education',
  'Phd Economics',
  'Ms Statistics',
  'Phd Public Health',
  'Ma Higher Education',
  'Phd Engineering',
  'Mph',
  'Ma Counseling',
  'Phd Computer Science',
  'Phd Psychology',
  'Ms Public Policy',
  'Ms Data Science',
  'Ms Computer Science'],
 'term_range': {'min': 'Fall 2018', 'max': 'Spring 2024'},
 'null_counts': {'student_id': 0,
  'last_name': 0,
  'first_name': 0,
  'academic_term': 0,
  'term_year': 0,
  'program': 0,
  'year_in_program': 0,
  'degree_type': 0,
  'pell_eligible': 140,
  'grant_amt': 498,
  'fellowship': 24,
  'TA_status': 143,
  'GSR_status': 144,
  'gpa': 11,
  'units_enrolled': 20,
  'stipend_amt': 199,
  'thesis_status': 171,
  'time_to_degree': 508,
  'employment_outcome': 504,
  'employer_type': 508,
  'salary_offer': 509,
  'job_title': 509,
  'industry': 509,
  'months_to_employment': 509,
  'gender': 0,
  'ethn

## analysis-ready metrics

In [20]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Derive analysis-ready metrics from the raw columns.

    This consolidates every transformation explored individually in the
    Step 3 cells above into a single callable function, so the full
    pipeline can be run end-to-end via main().
    """
    # Term label
    df["term_label"] = df["academic_term"].str.strip() + " " + df["term_year"].astype(str)

    # Support flags
    df["any_support"] = (df["TA_status"].fillna(False) |
                          df["GSR_status"].fillna(False))

    df["financial_aid_flag"] = (
        df["pell_eligible"].fillna(False) |
        (df["grant_amt"].fillna(0) > 0)
    )

    df["total_financial_support"] = (
        df["grant_amt"].fillna(0) + df["stipend_amt"].fillna(0)
    )

    # Academic risk
    df["gpa_risk_flag"] = df["gpa"] < 3.0

    # Thesis stage (ordinal)
    stage_order = {
        "Proposal": 1, "Approved": 2,
        "Writing": 3, "Defended": 4,
        "N/A": 0
    }
    df["thesis_stage_num"] = (
        df["thesis_status"].str.title().map(stage_order).fillna(0).astype(int)
    )

    # Degree type
    df["degree_program_type"] = np.where(
        df["program"].str.upper().str.startswith("PHD"),
        "PHD",
        "Masters"
    )

    # Cohort year
    df["cohort_year"] = df.groupby("student_id")["term_year"].transform("min")
    df["years_since_cohort"] = df["term_year"].astype(int) - df["cohort_year"]

    # Interaction: pell-eligible AND first-gen
    df["pell_x_first_gen"] = (
        df["pell_eligible"].fillna(False) & df["first_gen"].fillna(False)
    ).astype(int)

    # Career outcomes
    df["graduated_flag"] = df["time_to_degree"].notna()

    df["employed_flag"] = pd.NA
    df.loc[df["graduated_flag"] & (df["employment_outcome"].str.upper() == "EMPLOYED"), "employed_flag"] = 1
    df.loc[df["graduated_flag"] & (df["employment_outcome"].str.upper() != "EMPLOYED"), "employed_flag"] = 0
    df["employed_flag"] = df["employed_flag"].astype("Int64")

    df.loc[~df["graduated_flag"], "salary_offer"] = np.nan
    df["log_salary"] = np.log(df["salary_offer"].replace(0, np.nan))

    print(f"[ENGINEER] Feature engineering complete")
    return df

In [21]:
# orchestrate the pipeline
def main():
    print("=" * 55)
    print("  Graduate Student Data Cleaning Pipeline")
    print("=" * 55)

    df = load_raw(RAW_PATH)

    df = standardize_booleans(df)
    df = standardize_numerics(df)
    df = standardize_strings(df)
    df = engineer_features(df)

    # Save cleaned data
    CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(CLEAN_PATH, index=False)
    print(f"\n[SAVE] Clean data → {CLEAN_PATH}")

    # Save QA report
    report = build_quality_report(df)
    with open(QA_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print(f"[SAVE] QA report  → {QA_PATH}")

    print("\n[DONE] Pipeline complete.")
    print(f"       {report['total_rows']} rows | "
          f"{report['total_students']} students | "
          f"{report['pct_pell_eligible']}% Pell-eligible")


if __name__ == "__main__":
    main()

  Graduate Student Data Cleaning Pipeline
[LOAD] 550 x 28 columns loaded from /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/raw/grad_students_raw.csv
[STANDARDIZE] Boolean columns normalized
[STANDARDIZE] Numeric columns cast
[STANDARDIZE] String columns cleaned
[ENGINEER] Feature engineering complete

[SAVE] Clean data → /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/processed/grad_students_clean.parquet
[SAVE] QA report  → /Users/jenni/Desktop/PROJECTS/Data Scientist Python Projects/grad-outcomes-analysis/data/processed/data_quality_report.json

[DONE] Pipeline complete.
       550 rows | 95 students | 12.4% Pell-eligible
